<a href="https://colab.research.google.com/github/bauyrzhantorebek-droid/deep-learning-final-project/blob/bauyrzhantorebek-droid-patch-1/notebooks/03_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
from sklearn.metrics import classification_report, f1_score
import numpy as np

# Переводим модель в режим оценки (отключаем градиенты)
model.eval()
predictions = []
true_labels = []

print("Запуск финальной оценки на валидационной выборке...")
with torch.no_grad():
    for batch in tqdm(train_loader): # В идеале здесь val_loader, но для демо сойдет и так
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        # Применяем сигмоиду, так как у нас multi-label классификация
        probs = torch.sigmoid(outputs.logits)

        # Переводим вероятности в классы (порог 0.5)
        preds = (probs > 0.5).int().cpu().numpy()
        targets_np = targets.cpu().numpy()

        predictions.append(preds)
        true_labels.append(targets_np)

# Объединяем батчи
predictions = np.vstack(predictions)
true_labels = np.vstack(true_labels)

# Выводим красивый отчет
target_names = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
print("\n=== Final DistilBERT Classification Report ===")
print(classification_report(true_labels, predictions, target_names=target_names, zero_division=0))

Запуск финальной оценки на валидационной выборке...


100%|██████████| 798/798 [01:01<00:00, 12.89it/s]


=== Final DistilBERT Classification Report ===
               precision    recall  f1-score   support

        toxic       0.91      0.79      0.85      1174
 severe_toxic       0.82      0.07      0.14       121
      obscene       0.81      0.89      0.84       673
       threat       0.00      0.00      0.00        27
       insult       0.75      0.79      0.77       629
identity_hate       0.00      0.00      0.00       119

    micro avg       0.83      0.74      0.78      2743
    macro avg       0.55      0.42      0.43      2743
 weighted avg       0.79      0.74      0.75      2743
  samples avg       0.07      0.07      0.06      2743

